# Benchmark coverage and PULSE chronology
## tl;dr
PULSE has 12 past-only coefficient-fit folds: ratings 2014–2025, outcomes 2015–2026.
All three internal candidates score 14,439 identical games. These are reused development
seasons, not untouched confirmation. The old external comparison is not current PULSE,
and its internal RAPM updates do not preserve the claimed final common-player support.
The corrected benchmark uses 13,209 identical games from 2016–2026. PULSE RMSE is
13.7545, compared with 13.8644 for RAPM and 13.6791 for xRAPM.

## Context & Methods
This audit profiles existing local inputs and validates saved predictions. It does not
fit models, fetch data, open reserved 2027 outcomes, or change production ratings.
Years identify the year in which an NBA season ends.

### Key Assumptions
File coverage establishes availability, not point-in-time provenance for third-party
training. PIPM 2021 is partial (maximum 22 GP). MAMBA labels after 2024 are not usable
season panels. RAPTOR's historical and modern variants use different information.
The requested main panel uses saved CourtSignal PIPM/RAPTOR reconstructions.
RAPTOR's pooled training through 2022 makes its early results non-chronological.


In [1]:
from pathlib import Path
import hashlib, json
import pandas as pd
from IPython.display import display
from nba_impact.models.pulse_validation import load_pulse_validation
from nba_impact.api.web_snapshot import _pulse_evidence
from research.run_external_all_in_one_benchmark_v2 import read_xlsx_sheet, season_end

ROOT = Path.cwd()
DOWNLOADS = Path.home() / 'Downloads'
PULSE = ROOT / 'artifacts/models/pulse/pulse_canonical_v1_cd3c14750a'
OLD = ROOT / 'artifacts/research/external_all_in_one_benchmark/external_all_in_one_benchmark_v2_6a898e99d9'
sources = {}
coverage = []
def record(name, path, frame, year_column, off, defense, note='', stop=2026):
    sources[name] = {'file': str(path.relative_to(ROOT)) if path.is_relative_to(ROOT) else path.name,
                     'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}
    years = frame[year_column].map(season_end)
    valid = frame[off].notna() & frame[defense].notna() & years.le(stop)
    present = sorted(years[valid].unique().tolist())
    coverage.append({'model': name, 'first_rating_year': min(present),
                     'last_rating_year': max(present), 'seasons': len(present),
                     'rows': int(valid.sum()), 'note': note})


## Data
### Source coverage
No model is refit. Only non-null offensive and defensive ratings count.

In [2]:
inputs = [
 ('EPM', 'EPM_All_Seasons.csv', 'EPM_season', 'EPM_off', 'EPM_def'),
 ('LEBRON', 'lebron-data-2026-2025-2024-2023-2022-2021-2020-2019-2018-2017-2016-2015-2014-2013-2012-2011-2010.csv', 'Season', 'O-LEBRON', 'D-LEBRON'),
 ('PIPM', 'PIPM Player Finder through 2021 - Database.csv', 'Season', 'O-PIPM', 'D-PIPM'),
 ('RAPTOR modern', 'Data/modern_RAPTOR_by_player.csv', 'season', 'raptor_offense', 'raptor_defense'),
 ('RAPTOR latest', 'Data/latest_RAPTOR_by_player.csv', 'season', 'raptor_offense', 'raptor_defense'),
 ('RAPTOR historical', 'Data/historical_RAPTOR_by_player.csv', 'season', 'raptor_offense', 'raptor_defense'),
 ('MAMBA', 'MAMBAVALUES.xlsx - Sheet1.csv', 'Season', 'Offense', 'Defense'),
]
for name, filename, year, off, defense in inputs:
    path = DOWNLOADS / filename
    raw = pd.read_csv(path, low_memory=False)
    note = {'PIPM': '2021 is partial; use through 2020 for complete-season comparison',
            'MAMBA': 'Exclude malformed labels after 2024'}.get(name, '')
    record(name, path, raw, year, off, defense, note, 2024 if name == 'MAMBA' else 2026)
path = DOWNLOADS / 'DARKO - Daily Adjusted and Regressed Kalman Optimized projections - Full DPM History.csv'
darko = pd.read_csv(path, low_memory=False)
assert not darko.duplicated(['nba_id', 'season']).any()
assert darko.season.max() == 2026
record('DARKO DPM', path, darko, 'season', 'o_dpm', 'd_dpm',
       'Updated user-supplied CSV replaces the older workbook; per-player date is available.')
path = ROOT / 'research/rapm_lab/data/external/benchmark_20260903/annual_sources.parquet'
raw = pd.read_parquet(path)
for name, off, defense in [('BPM 2.0', 'bpm_offense', 'bpm_defense'), ('xRAPM', 'xrapm_offense', 'xrapm_defense')]:
    record(name, path, raw, 'season', off, defense)
for family, run, prefix in [('pipm', 'pipm_reconstruction_v1_e0625de5fe', 'pipm'),
                            ('raptor', 'raptor_reconstruction_v1_938d1becf9', 'raptor')]:
    path = ROOT / f'research/rapm_lab/outputs/{family}_reconstruction/{run}/reconstructions.parquet'
    record(f'CourtSignal {family.upper()} reconstruction', path, pd.read_parquet(path),
           'season', f'{prefix}_offense', f'{prefix}_defense', 'Saved reconstruction used as-is')
display(pd.DataFrame(coverage))


,model,first_rating_year,last_rating_year,seasons,rows,note
0,EPM,2002,2026,25,11452,
1,LEBRON,2010,2026,17,8715,
2,PIPM,1974,2021,48,19162,2021 is partial; use through 2020 for complete...
3,RAPTOR modern,2014,2022,9,4685,
4,RAPTOR latest,2023,2023,1,541,
5,RAPTOR historical,1977,2022,46,19159,
6,MAMBA,2015,2024,10,5275,Exclude malformed labels after 2024
7,DARKO DPM,1997,2026,30,15054,Updated user-supplied CSV replaces the older w...
8,BPM 2.0,2014,2026,13,6940,
9,xRAPM,1997,2026,30,14762,


In [3]:
pipm = pd.read_csv(DOWNLOADS / inputs[2][1], low_memory=False)
pipm = pipm[pipm.Season.isin(['2018-19', '2019-20', '2020-21'])]
display(pipm.groupby('Season').agg(rows=('Player','size'), max_games=('GP','max')))
mamba = pd.read_csv(DOWNLOADS / inputs[6][1])
display(mamba[mamba.Season.le(2026)].groupby('Season').size().rename('rows').to_frame())


,rows,max_games
Season,,
2018-19,530,105
2019-20,532,94
2020-21,474,22


,rows
Season,
2015,492
2016,476
2017,486
2018,540
2019,530
2020,528
2021,540
2022,605
2023,539


## Results
### Chronological predictions, not full-history display ratings

In [4]:
games, folds = load_pulse_validation(PULSE)
display(folds[['rating_season','outcome_season','training_start','training_end']].drop_duplicates())
display(games.groupby('candidate').agg(game_rows=('game_id','size'), seasons=('outcome_season','nunique')))
manifest = json.loads((PULSE/'run.json').read_text())
assert manifest['final_prior']['training_end'] == 2026
assert folds.training_end.lt(folds.rating_season).all()
assert games.outcome_season.eq(games.rating_season + 1).all()
public = json.loads((ROOT/'web/public/data/catalog.json').read_text())
checked_summary = _pulse_evidence(ROOT, manifest)['comparison']
assert checked_summary == public['methods']['pulse']['comparison']
print('All declared folds passed. The existing public internal summary is numerically unchanged.')
for name in ('run.json','validation_games.parquet','validation_folds.parquet','validation_priors.parquet'):
    path = PULSE/name
    sources['PULSE '+name] = {'file': str(path.relative_to(ROOT)), 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}


,rating_season,outcome_season,training_start,training_end
0,2014,2015,2005,2013
3,2015,2016,2005,2014
6,2016,2017,2005,2015
9,2017,2018,2005,2016
12,2018,2019,2005,2017
15,2019,2020,2005,2018
18,2020,2021,2005,2019
21,2021,2022,2005,2020
24,2022,2023,2005,2021
27,2023,2024,2005,2022


,game_rows,seasons
candidate,,
prior,14439,12
pulse,14439,12
rapm,14439,12


All declared folds passed. The existing public internal summary is numerically unchanged.


### What the old external table actually measured
The source below contains older Box15, rich-SPM and residual priors. It contains no
current canonical PULSE ratings. The runner intersects priors, then fits each internal
RAPM update on the full player matrix. Thus it shares games and prior coverage, not
identical final player support. Do not republish it under a current PULSE label.


In [5]:
old = pd.read_parquet(OLD/'ratings.parquet', columns=['candidate','rating_season'])
display(old.groupby('candidate').rating_season.agg(['min','max','nunique']))
assert 'pulse' not in set(old.candidate.str.lower())
print('Website pinned older external run:', public['validation']['metric_comparison']['run_id'])
print('Canonical recorded builder hash:', manifest['source_hashes']['builder'])
actual_builder = hashlib.sha256((ROOT/'src/nba_impact/models/canonical_pulse.py').read_bytes()).hexdigest()
print('Current builder hash:', actual_builder)
print('Builder source hash matches:', actual_builder == manifest['source_hashes']['builder'])


,min,max,nunique
candidate,,,
BPM 2.0,2017,2024,8
Box15,2017,2024,8
Box15 (2014+),2017,2024,8
DARKO DPM,2017,2024,8
DARKO preseason,2018,2018,1
Defense residual challenger,2017,2024,8
EPM,2017,2024,8
LEBRON,2017,2024,8
MAMBA,2017,2024,8


Website pinned older external run: external_all_in_one_benchmark_v2_eac56750f7
Canonical recorded builder hash: f05004fdbce7f850cae4ee0d3c11c646c3dda624990ad5a052bfcb80ed252964
Current builder hash: a11bb494762cf07b4696a359d2716df069ebe877b970e7dc2de3f529fc83ed9d
Builder source hash matches: False


### Corrected common-support comparison
The export verifier checks source and output hashes, chronological PULSE metadata,
exact replay, identical game keys and actual margins, equal final player inclusion masks,
zero coefficients for every excluded player, and independently recomputed RMSE.
It returns summaries only and does not write a website release when imported.


In [6]:
import runpy
builder = runpy.run_path(str(ROOT / 'web/scripts/build-external-benchmark.py'))
payload = builder['build_payload']()
run = builder['RUN']
for panel in payload['panels']:
    print(panel['scope'], panel['outcome_start'], panel['outcome_end'], panel['games'])
    display(pd.DataFrame(panel['rows'])[['candidate','aggregate_rmse','mean_correlation']])
display(pd.read_parquet(run/'paired_intervals.parquet').query("scope == 'main'"))
replay = json.loads((run/'run.json').read_text())['pulse_replay']
assert len(replay) == 11 and all(row['maximum_prediction_difference'] == 0 for row in replay)
print('All 11 PULSE folds reproduced saved predictions exactly before masking and centering.')


main 2016 2026 13209


,candidate,aggregate_rmse,mean_correlation
0,xRAPM,13.679103,0.383042
1,EPM,13.683575,0.378222
2,DARKO DPM,13.705633,0.365755
3,LEBRON,13.739062,0.358524
4,PULSE,13.754534,0.358489
5,BPM 2.0,13.849659,0.370974
6,RAPM,13.864354,0.336369
7,CourtSignal RAPTOR reconstruction,13.869518,0.372908
8,CourtSignal PIPM reconstruction,14.052890,0.327149


with_mamba 2016 2025 11979


,candidate,aggregate_rmse,mean_correlation
0,MAMBA,13.469676,0.381378
1,xRAPM,13.538913,0.380442
2,EPM,13.547438,0.375092
3,DARKO DPM,13.547499,0.364380
4,LEBRON,13.582587,0.356480
5,PULSE,13.593696,0.357576
6,RAPM,13.674860,0.339293
7,CourtSignal RAPTOR reconstruction,13.717152,0.370842
8,BPM 2.0,13.738868,0.365559
9,CourtSignal PIPM reconstruction,13.900758,0.326381


,scope,other,pulse_minus_other_rmse,lower_95,upper_95
0,main,BPM 2.0,-0.095125,-0.152052,-0.037752
1,main,CourtSignal PIPM reconstruction,-0.298356,-0.340623,-0.253891
2,main,CourtSignal RAPTOR reconstruction,-0.114984,-0.169462,-0.060520
3,main,DARKO DPM,0.048901,0.007894,0.089432
4,main,EPM,0.070959,0.033440,0.109948
5,main,LEBRON,0.015472,-0.026694,0.055637
6,main,Normal RAPM,-0.109820,-0.134695,-0.086238
7,main,xRAPM,0.075430,0.028123,0.122446


All 11 PULSE folds reproduced saved predictions exactly before masking and centering.


## Takeaways
- The main panel tests ratings 2015–2025 on 2016–2026 games, using identical final
  scored player support for all nine models. MAMBA ends with 2025 outcomes.
- PULSE beats RAPM by 0.1098 RMSE. The paired 95% interval is -0.1347 to -0.0862.
  xRAPM, EPM and DARKO have lower error; the gap to LEBRON is inconclusive.
- PULSE uses past-only prior fits. All 11 saved predictions replayed exactly before
  benchmark normalization. Final full-history display ratings never enter the test.
- CourtSignal RAPTOR and PIPM remain as-is. RAPTOR's pooled 2014–2022 training
  prevents a clean out-of-time claim for its early results. Source-author metric
  files do not certify original training cutoffs or point-in-time publication.
- Historical outcomes are selection-exposed and predictions use observed future
  lineups. This is a matched-support diagnostic, not untouched confirmation.
- Scores use official final margins and a common missing-player/baseline policy.
  They differ from the original unmasked internal PULSE table. No deployment occurred.


In [7]:
audit = ROOT/'research/audits/benchmark_coverage_20260903'
(audit/'source_coverage.json').write_text(json.dumps({'coverage': coverage, 'sources': sources}, indent=2))


5450